# TFG — StyleGAN2-ADA condicional (HAM10000) · Corrida diagnóstica en Kaggle

**Nicolás Lerible García — F4 (entrenamiento).** Primera corrida larga real sobre plataforma externa.

**Configuración fijada para esta corrida:**

| Parámetro | Valor | Origen de la decisión |
|---|---|---|
| Plataforma | Kaggle, GPU P100 16 GB | esquema de 4 capas (21/07/2026) |
| Dataset | `ham7_train_256.zip` (7995 img, 256×256, 7 clases) | F3 (30/07/2026) |
| Muestreo | `--sampling-alpha=-0.5` (`p_i ∝ n_i^α`) | groundwork F4 (30/07/2026) |
| Config | `--cfg=auto` → batch total 16 (batch_gpu=16) | reproduce el smoke run local validado |
| ADA | `--aug=ada --target=0.6 --augpipe=bgc` (sin cutout) | decisiones 21/07 y 30/07/2026 |
| Métricas | `--metrics=fid50k_full` (global) | FID+KID por clase van post-hoc (14/07 + hoy) |
| Presupuesto | ~5000 kimg (diagnóstico) | decisión de hoy |
| Snapshots | `--snap=25` → checkpoint cada 100 kimg | seguridad de `--resume` en sesiones cortas |
| Semilla | `--seed=42` | consistente con el split |

> **Antes de ejecutar** (barra derecha de Kaggle):
> 1. **Settings → Accelerator = GPU P100**.
> 2. **Settings → Internet = On** (necesario para pip y para descargar el detector Inception del FID).
> 3. **Add Input**: adjunta tus dos datasets de Kaggle — el repo parcheado y el zip de entrenamiento (ver celda de configuración).

In [ ]:
# 0) Comprobación de GPU y entorno
!nvidia-smi
import sys, torch
print('Python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 1. Preparativos previos (una sola vez, fuera del notebook)

Sube a Kaggle **dos Datasets** (Create → New Dataset):

1. **`stylegan2-ada-tfg`** — tu repositorio *ya parcheado* (con `training/weighted_sampler.py` y el `--sampling-alpha` aplicado por `scripts/patch_sampling_alpha.py`). Comprime la carpeta `stylegan2-ada-pytorch/` **sin** el `venv`, sin `.git` y sin datos, y súbela.
2. **`ham7-train-256`** — sube `ham7_train_256.zip` tal cual (Kaggle NO lo descomprime; `train.py` lee el zip directamente).

> Subir el repo como Dataset es preferible a `git clone` de tu repo privado: no expone tokens y queda versionado/reproducible. El zip de máscaras no hace falta aquí (las máscaras son para evaluación en F8, no para entrenar).

Ajusta los *slugs* de abajo a los nombres exactos que te asigne Kaggle (aparecen bajo `/kaggle/input/...`).

In [ ]:
# 2) Configuración central — EDITA solo esta celda
import os

# --- Rutas de entrada (ajusta a tus slugs reales de /kaggle/input) ---
REPO_SRC = '/kaggle/input/stylegan2-ada-tfg/stylegan2-ada-pytorch'   # repo parcheado
DATA_ZIP = '/kaggle/input/ham7-train-256/ham7_train_256.zip'         # dataset de entrenamiento

# --- Rutas de trabajo (persisten al hacer 'Save Version') ---
REPO_DIR = '/kaggle/working/stylegan2-ada-pytorch'
OUTDIR   = '/kaggle/working/experimentos'

# --- Hiperparámetros de la corrida (fijados) ---
CFG          = 'auto'          # -> batch total 16 en 1 GPU a 256px (== smoke run local)
COND         = 1              # dataset condicional de 7 clases
MIRROR       = 1              # x-flip a nivel dataset (sin orientación canónica en dermatoscopia)
AUG          = 'ada'
TARGET       = 0.6            # ada_target inicial (rango de barrido 0.5-0.7)
AUGPIPE      = 'bgc'          # sin cutout
SAMPLING_A   = -0.5           # tu flag custom: p_i proporc. n_i^alpha
METRICS      = 'fid50k_full'  # FID global; KID/PPL por clase -> post-hoc
SEED         = 42
SNAP         = 25            # snapshot cada 25 ticks = 100 kimg
WORKERS      = 2

# Presupuesto de ESTA sesión en kimg (no el total del diagnóstico).
# ~50 s/kimg -> 800 kimg ~= 11 h, holgado bajo el límite de sesión de Kaggle.
KIMG_SESSION = 800

assert os.path.isdir(REPO_SRC), f'No encuentro el repo en {REPO_SRC} — revisa Add Input y el slug'
assert os.path.isfile(DATA_ZIP), f'No encuentro el zip en {DATA_ZIP} — revisa Add Input y el slug'
print('OK: repo y dataset localizados.')

In [ ]:
# 3) Copiar el repo a /kaggle/working (escribible) e instalar dependencias
import shutil, os
if not os.path.isdir(REPO_DIR):
    shutil.copytree(REPO_SRC, REPO_DIR)
print('Repo en:', REPO_DIR)

# Dependencias de tiempo de ejecución del repo. Los kernels CUDA (bias_act, upfirdn2d)
# se COMPILAN con ninja la primera vez que se usan (tardan unos minutos en la 1a corrida).
!pip -q install ninja pyspng imageio-ffmpeg==0.4.3
print('Dependencias instaladas.')

In [ ]:
# 4) Verificaciones de cordura ANTES de gastar cuota de GPU
# 4a) Que tu parche del sampler está presente en este repo
!grep -n 'sampling-alpha\|sampling_alpha' {REPO_DIR}/train.py | head
print('---')
!ls {REPO_DIR}/training/weighted_sampler.py && echo 'weighted_sampler.py OK'
print('--- clases del dataset.json ---')
# 4b) Que el zip declara las 7 clases
import zipfile, json
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open('dataset.json') as f:
        dj = json.load(f)
labels = [l[1] for l in dj['labels']]
print('N imagenes etiquetadas:', len(labels))
print('N clases distintas   :', len(set(labels)), '(esperado 7)')

In [ ]:
# 5) DRY-RUN: valida toda la configuración sin entrenar (no consume tiempo de GPU real)
cmd = f'''cd {REPO_DIR} && python train.py \\
  --outdir={OUTDIR} \\
  --data={DATA_ZIP} \\
  --gpus=1 --cfg={CFG} --cond={COND} --mirror={MIRROR} \\
  --aug={AUG} --target={TARGET} --augpipe={AUGPIPE} \\
  --sampling-alpha={SAMPLING_A} \\
  --metrics={METRICS} \\
  --kimg={KIMG_SESSION} --snap={SNAP} --seed={SEED} --workers={WORKERS} \\
  --dry-run'''
print(cmd)
!{cmd}

## 6. Lanzamiento — SESIÓN 1 (desde cero)

Revisa la ficha que imprime el `--dry-run` (resolución 256, `label_dim=7`, `batch_gpu=16`, `augpipe=bgc`, `ada_target=0.6`). Si cuadra, ejecuta la celda siguiente.

El entrenamiento crea `…/experimentos/00000-ham7_train_256-...`. Ve mirando la salida: cada *tick* imprime `sec/kimg`, `augment` (= p de ADA) y, en cada snapshot, el FID.

In [ ]:
# 6) SESIÓN 1 — arranque desde cero (NO usa --resume)
cmd = f'''cd {REPO_DIR} && python train.py \\
  --outdir={OUTDIR} \\
  --data={DATA_ZIP} \\
  --gpus=1 --cfg={CFG} --cond={COND} --mirror={MIRROR} \\
  --aug={AUG} --target={TARGET} --augpipe={AUGPIPE} \\
  --sampling-alpha={SAMPLING_A} \\
  --metrics={METRICS} \\
  --kimg={KIMG_SESSION} --snap={SNAP} --seed={SEED} --workers={WORKERS}'''
print(cmd)
!{cmd}

## 7. Guardar los checkpoints al terminar la sesión

Cuando la sesión vaya a acabar (por límite de tiempo o cuando tú la pares):

1. **Save Version** (botón arriba a la derecha) → *Save & Run All* o *Quick Save*. Esto persiste `/kaggle/working/` como **Output** del notebook.
2. Entra en la pestaña **Output** de la versión guardada, localiza el `…/experimentos/00000-.../network-snapshot-XXXXXX.pkl` más reciente.
3. Crea (o **versiona**) un Dataset de Kaggle a partir de ese Output — llámalo `tfg-ckpts`. Ese Dataset será el input de resume de la próxima sesión.

> **Importante sobre `--resume` en este repo:** restaura **solo** los pesos de `G`, `D` y `G_ema`. **No** restaura el estado del optimizador, ni el `p` de ADA, ni el contador de kimg. Consecuencias prácticas: el eje de kimg y la numeración de snapshots **reinician a 0** en cada sesión (lleva la cuenta acumulada a mano), y el `p` de ADA se re-adapta desde 0 en los primeros ~cientos de kimg tras cada resume. Es asumible para un diagnóstico; si el trasiego multi-sesión se hace pesado, se puede parchear un resume de estado completo (~30 líneas).

In [ ]:
# 7b) Inventario de snapshots generados en esta sesión (para saber cuál usar en el resume)
import glob, os
for pkl in sorted(glob.glob(f'{OUTDIR}/*/network-snapshot-*.pkl')):
    mb = os.path.getsize(pkl)/1e6
    print(f'{mb:8.1f} MB  {pkl}')
print('\n→ El de mayor número es el último checkpoint de esta sesión.')

## 8. Lanzamiento — SESIONES 2, 3, … (con `--resume`)

En cada sesión nueva:

1. Adjunta como Input el Dataset `tfg-ckpts` con el último `.pkl`.
2. Pon la ruta en `RESUME_PKL` abajo.
3. Ejecuta las celdas 0, 2, 3 y luego **esta** (no la celda 6).

Suma los kimg de cada sesión para saber cuánto llevas del presupuesto de ~5000 (p. ej. 800 × 6 ≈ 4800).

In [ ]:
# 8) SESIONES 2+ — continuar desde el último checkpoint
RESUME_PKL = '/kaggle/input/tfg-ckpts/network-snapshot-000800.pkl'  # <-- EDITA a tu último pkl
import os
assert os.path.isfile(RESUME_PKL), f'No encuentro {RESUME_PKL} — revisa Add Input y la ruta'

cmd = f'''cd {REPO_DIR} && python train.py \\
  --outdir={OUTDIR} \\
  --data={DATA_ZIP} \\
  --gpus=1 --cfg={CFG} --cond={COND} --mirror={MIRROR} \\
  --aug={AUG} --target={TARGET} --augpipe={AUGPIPE} \\
  --sampling-alpha={SAMPLING_A} \\
  --metrics={METRICS} \\
  --kimg={KIMG_SESSION} --snap={SNAP} --seed={SEED} --workers={WORKERS} \\
  --resume={RESUME_PKL}'''
print(cmd)
!{cmd}

## 9. Troubleshooting (errores frecuentes de StyleGAN2-ADA en entornos nuevos)

- **`AttributeError: module 'distutils' has no attribute 'version'`** (al compilar los kernels): `!pip install setuptools==59.5.0` y reinicia el kernel. Típico con torch 1.x; con el torch de stock de Kaggle (2.x) no suele hacer falta.
- **`ninja: build stopped` / `nvcc not found`**: falta el toolchain CUDA. Verifica `!which nvcc`; en Kaggle suele estar. Reintenta tras `!pip install ninja`.
- **`RuntimeError: CUDA out of memory`**: no debería pasar con `cfg=auto` (batch_gpu=16, ~5 GB) en P100 16 GB. Si ocurre, baja `--batch` a 8.
- **El FID se queda colgado la 1ª vez**: está descargando `inception-2015-12-05.pkl` (Internet debe estar ON). Solo ocurre una vez por sesión.
- **Avisos de `conv2d_gradfix`/`grid_sample_gradfix` deshabilitados**: son esperables con torch nuevo; el repo cae a la ruta estándar de PyTorch y sigue entrenando correctamente.

## 10. Después de la corrida diagnóstica (siguientes pasos)

1. **Curvas** con `scripts/plot_training_stats.py` sobre los `stats.jsonl` de cada segmento; al concatenar sesiones, desplaza el kimg por el acumulado de cada resume (el contador reinicia en 0).
2. **FID + KID por clase** (decisión 14/07) — análisis **post-hoc** sobre los mejores checkpoints, evaluando cada clase por separado; especialmente relevante para `vasc` y `df`.
3. **PPL** post-hoc con `calc_metrics.py --metrics=ppl2_wend` (decisión 21/07), overhead cero sobre el entrenamiento.
4. **Galería por clase** con `generate.py` para las primeras imágenes generadas — lo que el tutor pidió ver antes de la próxima reunión.